# Importation et chargement du modèle

In [ ]:
import torch
import random
import src.ADG as adg

In [2]:
# Chargement du modèle
device = "cuda" if torch.cuda.is_available() else "cpu"
model, tokenizer = adg.load_model("Qwen/Qwen2.5-0.5B", device)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1429.55it/s]


Loaded Qwen/Qwen2.5-0.5B on cpu (vocab size: 151643)


# Fonctions de visualisation des résultats

In [3]:
def print_text_info(text):
    print("INFOS")
    print(f"label            = {text.steg_label}")
    print(f"payload_size     = {text.payload_size}")
    print(f"nb_tokens        = {len(text.tokens)}")
    print(f"embedding_rate   = {text.embedding_rate}")
    print(f"cause de l'arret = {text.stop_reason}")

def test_adg(taille_budget , max_tokens, verbose = True):
    payload = random.choices([0, 1], k=taille_budget )
    print(f" budget : {taille_budget} bits")
    print(f" nb max de tokens : {max_tokens} \n")
    # On teste la génération de cover
    print("Test GENERATION COVER")
    cover = adg.ADG_generate_cover(prompt_ids, 
                                bit_budget =taille_budget,
                                max_tokens=max_tokens,
                                config=config )
    if verbose ==  True:
        print(f"{prompt} <---> {cover.text} \n") 
    print_text_info(cover) 
    print("\n"+ 50*"*" +"\n")

    # On teste la génération de Stego
    #Stego : Alice genère une séquence dissimulant le payload
    print("Test GENERATION STEGO ")
    stego = adg.ADG_encode(prompt_ids,
                        payload, 
                        config, 
                        max_tokens)
    if verbose ==  True: 
        print(f"{prompt} <---> {stego.text} \n") 
    print_text_info(stego) 
    #Et la reconstruction
    # Bob reconstitue le payload, il rejoue le routage pour retrouver les bits.
    print("\n Reconstruction? ")
    recovered = adg.ADG_decode(prompt_ids, stego.tokens, taille_budget, config)
   
    if stego.stop_reason is adg.StopReason.BIT_BUDGET_REACHED:
        ok = recovered == payload          # complet : tout doit correspondre
    else:
        ok = recovered == payload[:stego.payload_size]   # partiel
    print("reconstruction correcte :", ok)
  

# Configuration d'ADG 

In [4]:
# Configuration ADG
config = adg.ADGConfig(model=model, 
                       tokenizer=tokenizer,
                       device=device, 
                       temperature=1, 
                       top_k=50)

print(f"liste des index de EOS : {config.eos_token_ids}")

liste des index de EOS : [151643]


# Expériences de vérification du code ADG

In [5]:
SEED = 1976
random.seed(SEED)
torch.manual_seed(SEED)

prompt = "Little white rabbit, are you really going to"
prompt_ids = adg.encode_prompt(prompt, config)
print(f"PROMPT : {prompt}")

PROMPT : Little white rabbit, are you really going to


## Expérience 1 :  `stop_reason = BIT_BUDGET_REACHED`

In [6]:
# On choisit le taille_budget petit et max_tokens grand
taille_budget = 100
max_tokens = 10000

test_adg(taille_budget, max_tokens, verbose = True)

 budget : 100 bits
 nb max de tokens : 10000 

Test GENERATION COVER


Little white rabbit, are you really going to <--->  do it? I was in the village that the fairy was visiting and I found a beautiful red flower that lay on the ground beside the riverbank. Then I picked some small pieces of moss 

INFOS
label            = cover
payload_size     = 103
nb_tokens        = 38
embedding_rate   = 2.710526315789474
cause de l'arret = bit_budget_reached

**************************************************

Test GENERATION STEGO 
Little white rabbit, are you really going to <--->  the mall!? Little blue rabbit, can you help me please!
(请为男孩们加油) ______________
I can 

INFOS
label            = stego
payload_size     = 100
nb_tokens        = 26
embedding_rate   = 3.8461538461538463
cause de l'arret = bit_budget_reached

 Reconstruction? 
reconstruction correcte : True


## Expérience 2 :  `stop_reason = MAX_TOKENS_REACHED`

In [7]:
# On choisit le taille_budget grand et max_tokens petit
taille_budget = 1000
max_tokens = 10

test_adg(taille_budget, max_tokens)

 budget : 1000 bits
 nb max de tokens : 10 

Test GENERATION COVER
Little white rabbit, are you really going to <--->  be my friend?  I'm not going to 

INFOS
label            = cover
payload_size     = 20
nb_tokens        = 10
embedding_rate   = 2.0
cause de l'arret = max_tokens_reached

**************************************************

Test GENERATION STEGO 
Little white rabbit, are you really going to <--->  the park for fun?
A. Are you going 

INFOS
label            = stego
payload_size     = 19
nb_tokens        = 10
embedding_rate   = 1.9
cause de l'arret = max_tokens_reached

 Reconstruction? 
reconstruction correcte : True


## Expérience 3 :  `stop_reason = EOS_EMITTED`

In [8]:
# On choisit le taille_budget grand et max_tokens grand
taille_budget = 10000
max_tokens = 10000

test_adg(taille_budget, max_tokens, verbose = False)

 budget : 10000 bits
 nb max de tokens : 10000 

Test GENERATION COVER
INFOS
label            = cover
payload_size     = 2011
nb_tokens        = 1044
embedding_rate   = 1.9262452107279693
cause de l'arret = eos_emitted

**************************************************

Test GENERATION STEGO 
INFOS
label            = stego
payload_size     = 424
nb_tokens        = 282
embedding_rate   = 1.50354609929078
cause de l'arret = eos_emitted

 Reconstruction? 
reconstruction correcte : True
